# Data cleaning project: marketing campaign data
In this notebook I clean `marketing_campaign_data_messy.csv` one step at a time.
For each problem I first **show** it, then **fix** it, then **check** the fix.

In [1]:
import pandas as pd
import numpy as np

# loading messy data
df = pd.read_csv("marketing_campaign_data_messy.csv")
print(f"loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

loaded dataset: 2020 rows, 12 columns


In [2]:
df.head()

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0    Campaign_ID   2020 non-null   str    
 1   Campaign_Name  2020 non-null   str    
 2   Start_Date     2020 non-null   str    
 3   End_Date       2020 non-null   str    
 4   Channel        1919 non-null   str    
 5   Impressions    2020 non-null   int64  
 6   Clicks         2020 non-null   int64  
 7   Spend          2020 non-null   str    
 8   Conversions    1820 non-null   float64
 9   Active         2020 non-null   str    
 10  Clicks         40 non-null     float64
 11  Campaign_Tag   2020 non-null   str    
dtypes: float64(2), int64(2), str(8)
memory usage: 338.2 KB


## 1. Column names
The names have extra spaces (`' Campaign_ID '`, `'Clicks '`) and mixed case.
There are also **two** clicks columns: `'Clicks '` and `'Clicks'`. The second one is almost empty, and its values
are bigger than the impressions, which is impossible, so it's junk. I drop it before renaming, because otherwise both columns would end up being called `clicks`.

In [4]:
print(df.columns.tolist())

print("non-empty values in the 2nd 'Clicks' column:", df['Clicks'].notna().sum())
df = df.drop(columns=['Clicks'])

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
# str.strip() removes spaces at the start and end of each column name
# str.lower() converts all column names to lower case
# str.replace(' ', '_') replaces any remaining spaces with underscores
print('FIX APPLIED')
print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
non-empty values in the 2nd 'Clicks' column: 40
FIX APPLIED
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'campaign_tag']


## 2. Duplicate rows
Some campaigns show up more than once. `campaign_id` should be unique, so I keep the first row for each id.

In [5]:
print("duplicate campaign_ids:", df['campaign_id'].duplicated().sum())
df = df.drop_duplicates(subset='campaign_id', keep='first').reset_index(drop=True)
print('FIX APPLIED')
print("duplicate campaign_ids:", df['campaign_id'].duplicated().sum(), "| rows left:", len(df))

duplicate campaign_ids: 20
FIX APPLIED
duplicate campaign_ids: 0 | rows left: 2000


## 3. Spend: remove `$` and make it numeric

In [6]:
dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask, ['campaign_id', 'spend']].head(3))

# remove everything that isn't a digit, a dot or a minus sign
df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')

print('FIX APPLIED')
print(df.loc[dirty_spend_mask, ['campaign_id', 'spend']].head(3))
print("spend dtype:", df['spend'].dtype)

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
FIX APPLIED
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22
spend dtype: float64


### 3b. Spend: impossible values
Some spends are **negative** (the amounts look normal, so the minus sign is a typo), and some are exactly
**500000**, which is a placeholder (it would mean paying hundreds of dollars per click, when normal is about $0.5–2).

In [7]:
print("negative spend:", (df['spend'] < 0).sum())
print(df.loc[df['spend'] == 500000, ['campaign_id', 'spend', 'clicks']])

df['spend'] = df['spend'].abs()

# replace the placeholder with: clicks * normal (median) cost per click
placeholder = df['spend'] == 500000
normal_cpc = (df.loc[~placeholder, 'spend'] / df.loc[~placeholder, 'clicks']).median()
print("median cost per click:", round(normal_cpc, 2))
df.loc[placeholder, 'spend'] = (df.loc[placeholder, 'clicks'] * normal_cpc).round(2)

print('FIX APPLIED')
print("negative spend:", (df['spend'] < 0).sum(), "| placeholder spend:", (df['spend'] == 500000).sum())
print(df.loc[placeholder, ['campaign_id', 'spend', 'clicks']])

negative spend: 19
     campaign_id     spend  clicks
789    CMP-00790  500000.0    4272
1460   CMP-01461  500000.0    1193
1718   CMP-01719  500000.0    1585
1754   CMP-01755  500000.0    1392
1781   CMP-01782  500000.0     193
median cost per click: 1.24
FIX APPLIED
negative spend: 0 | placeholder spend: 0
     campaign_id    spend  clicks
789    CMP-00790  5292.57    4272
1460   CMP-01461  1478.01    1193
1718   CMP-01719  1963.65    1585
1754   CMP-01755  1724.55    1392
1781   CMP-01782   239.11     193


## 4. Channel: fix spelling mistakes

In [8]:
print(df['channel'].unique())

cleanup_map = {
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram',
    'Gogle': 'Google Ads',
    'Tik_Tok': 'TikTok',
    'E-mail': 'Email',
    'N/A': np.nan}
df['channel'] = df['channel'].replace(cleanup_map)
print("FIX APPLIED")
print(df['channel'].unique())

<ArrowStringArray>
[    'TikTok',   'Facebook',      'Email',  'Instagram', 'Google Ads',
     'E-mail',          nan,      'Gogle',    'Tik_Tok',    'Facebok',
 'Insta_gram']
Length: 11, dtype: str
FIX APPLIED
<ArrowStringArray>
['TikTok', 'Facebook', 'Email', 'Instagram', 'Google Ads', nan]
Length: 6, dtype: str


Some channels are missing. I don't want to guess them, so I label them `'Unknown'`.

In [9]:
print("missing channels:", df['channel'].isna().sum())
df['channel'] = df['channel'].fillna('Unknown')
print("FIX APPLIED")
print(df['channel'].value_counts())

missing channels: 100
FIX APPLIED
channel
TikTok        412
Facebook      401
Email         375
Instagram     358
Google Ads    354
Unknown       100
Name: count, dtype: int64


## 5. Campaign tag
The tag is just the first two letters of the channel, but it has bad values like `INVALID`, `XX` and `E-`.
Since the channel is clean now, I rebuild the tag from it.

In [10]:
print(df['campaign_tag'].value_counts())

tag_map = {'TikTok': 'TI', 'Facebook': 'FA', 'Instagram': 'IN',
           'Google Ads': 'GO', 'Email': 'EM', 'Unknown': 'UN'}
df['campaign_tag'] = df['channel'].map(tag_map)

print("FIX APPLIED")
print(df['campaign_tag'].value_counts())

campaign_tag
TI         397
FA         387
IN         342
EM         331
GO         331
INVALID    101
XX          92
E-          19
Name: count, dtype: int64
FIX APPLIED
campaign_tag
TI    412
FA    401
EM    375
IN    358
GO    354
UN    100
Name: count, dtype: int64


## 6. Dates
`start_date` has three formats: `2023-11-24 00:00:00`, `2023-10-30` and `22/02/2023` (day/month/year).
I parse the ISO dates first, then parse whatever failed as day/month/year.

In [11]:
print(df['start_date'].sample(8, random_state=1).tolist())

start_iso = pd.to_datetime(df['start_date'].str[:10], format='%Y-%m-%d', errors='coerce')
start_dmy = pd.to_datetime(df['start_date'], format='%d/%m/%Y', errors='coerce')
df['start_date'] = start_iso.fillna(start_dmy)
df['end_date'] = pd.to_datetime(df['end_date'], format='%Y-%m-%d', errors='coerce')

print("FIX APPLIED")
print("unparsed start dates:", df['start_date'].isna().sum(), "| unparsed end dates:", df['end_date'].isna().sum())
print(df[['start_date', 'end_date']].dtypes)

['2023-08-30 00:00:00', '2023-08-31 00:00:00', '2023-06-26 00:00:00', '2023-03-31 00:00:00', '2023-06-05 00:00:00', '2023-08-17 00:00:00', '2023-12-30 00:00:00', '2023-09-27']
FIX APPLIED
unparsed start dates: 0 | unparsed end dates: 0
start_date    datetime64[us]
end_date      datetime64[us]
dtype: object


A campaign can't end before it starts. Let's check:

In [12]:
bad_dates = df['end_date'] < df['start_date']
print("campaigns ending before they start:", bad_dates.sum())
df.loc[bad_dates, ['campaign_id', 'start_date', 'end_date']].head()

campaigns ending before they start: 78


,campaign_id,start_date,end_date
23,CMP-00024,2023-05-06,2023-05-01
54,CMP-00055,2023-09-01,2023-08-27
71,CMP-00072,2023-02-01,2023-01-27
124,CMP-00125,2023-04-01,2023-02-02
156,CMP-00157,2023-12-06,2023-12-01


In [13]:
# these are most likely start/end typed in the wrong order, so I swap them
df.loc[bad_dates, ['start_date', 'end_date']] = df.loc[bad_dates, ['end_date', 'start_date']].values
print("FIX APPLIED")
print("campaigns ending before they start:", (df['end_date'] < df['start_date']).sum())

FIX APPLIED
campaigns ending before they start: 0


## 7. Active: one consistent True/False
The column mixes `Y`, `Yes`, `True`, `1` with `No`, `False`, `0`.

In [14]:
print(df['active'].unique())

active_map = {'Y': True, 'Yes': True, 'True': True, '1': True,
              'No': False, 'False': False, '0': False}
df['active'] = df['active'].astype(str).str.strip().map(active_map)

print("FIX APPLIED")
print(df['active'].value_counts(dropna=False))

<ArrowStringArray>
['Y', '0', 'No', 'True', 'Yes', '1', 'False']
Length: 7, dtype: str
FIX APPLIED
active
True     1182
False     818
Name: count, dtype: int64


## 8. Conversions: missing values
200 rows have no conversions. Filling them with 0 would pretend nothing was sold, so I fill each missing value
with the median **conversion rate** (conversions / clicks) of its channel times its clicks.

In [15]:
print("missing conversions:", df['conversions'].isna().sum())

conv_rate = (df['conversions'] / df['clicks']).groupby(df['channel']).transform('median')
df['conversions'] = df['conversions'].fillna((conv_rate * df['clicks']).round())
df['conversions'] = df['conversions'].astype(int)

print("FIX APPLIED")
print("missing conversions:", df['conversions'].isna().sum())

missing conversions: 200
FIX APPLIED
missing conversions: 0


## 9. Sanity checks
Clicks can't be more than impressions, and conversions can't be more than clicks.

In [16]:
print("clicks > impressions:", (df['clicks'] > df['impressions']).sum())
print("conversions > clicks:", (df['conversions'] > df['clicks']).sum())
print("negative numbers:", (df[['impressions', 'clicks', 'spend', 'conversions']] < 0).sum().sum())

clicks > impressions: 0
conversions > clicks: 0
negative numbers: 0


## 10. New useful columns

In [17]:
df['duration_days'] = (df['end_date'] - df['start_date']).dt.days
df['ctr'] = (df['clicks'] / df['impressions']).round(4)           # click-through rate
df['cpc'] = (df['spend'] / df['clicks']).round(2)                 # cost per click
df['conversion_rate'] = (df['conversions'] / df['clicks']).round(4)
df['cost_per_conversion'] = (df['spend'] / df['conversions'].replace(0, np.nan)).round(2)
df.head()

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,duration_days,ctr,cpc,conversion_rate,cost_per_conversion
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20,True,TI,19,0.0117,0.52,0.1015,5.14
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1,False,FA,6,0.0161,0.81,0.0333,24.33
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51,False,EM,7,0.0108,1.57,0.0605,25.95
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135,True,TI,4,0.0361,1.08,0.0669,16.15
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30,True,FA,1,0.0233,1.49,0.1775,8.41


## Final result

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   campaign_id          2000 non-null   str           
 1   campaign_name        2000 non-null   str           
 2   start_date           2000 non-null   datetime64[us]
 3   end_date             2000 non-null   datetime64[us]
 4   channel              2000 non-null   str           
 5   impressions          2000 non-null   int64         
 6   clicks               2000 non-null   int64         
 7   spend                2000 non-null   float64       
 8   conversions          2000 non-null   int64         
 9   active               2000 non-null   bool          
 10  campaign_tag         2000 non-null   str           
 11  duration_days        2000 non-null   int64         
 12  ctr                  2000 non-null   float64       
 13  cpc                  2000 non-null   float64

In [19]:
print("missing values per column:")
print(df.isna().sum())

missing values per column:
campaign_id            0
campaign_name          0
start_date             0
end_date               0
channel                0
impressions            0
clicks                 0
spend                  0
conversions            0
active                 0
campaign_tag           0
duration_days          0
ctr                    0
cpc                    0
conversion_rate        0
cost_per_conversion    1
dtype: int64


In [20]:
summary = df.groupby('channel').agg(
    campaigns=('campaign_id', 'count'),
    total_spend=('spend', 'sum'),
    total_conversions=('conversions', 'sum'),
    avg_ctr=('ctr', 'mean'),
)
summary['cost_per_conversion'] = (summary['total_spend'] / summary['total_conversions']).round(2)
summary.sort_values('cost_per_conversion').round(3)

,campaigns,total_spend,total_conversions,avg_ctr,cost_per_conversion
channel,,,,,
Unknown,100,174937.32,19175,0.031,9.12
Instagram,358,635937.32,66713,0.030,9.53
Facebook,401,790490.60,77886,0.030,10.15
Google Ads,354,685842.46,66939,0.030,10.25
TikTok,412,773713.77,75340,0.030,10.27
Email,375,703874.58,68099,0.029,10.34


In [21]:
df.to_csv("marketing_campaign_data_clean.csv", index=False)
print(f"saved clean dataset: {df.shape[0]} rows, {df.shape[1]} columns")

saved clean dataset: 2000 rows, 16 columns
